# 06 — Market Breadth Dashboard

## Why this project exists

A market index can rise while fewer stocks or sectors participate. Traders often ask: **"Is this rally broad or narrow?"**

This notebook uses sector ETFs as a public-data proxy for market breadth. The goal is not to replicate institutional breadth data, but to demonstrate the dashboard logic.

In [ ]:
!pip install yfinance plotly -q

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.express as px

pd.options.display.float_format = "{:,.4f}".format

## 1. Download sector ETF data

I use SPDR sector ETFs to approximate sector-level participation.

In [ ]:
sectors = {
    "XLC": "Communication Services",
    "XLY": "Consumer Discretionary",
    "XLP": "Consumer Staples",
    "XLE": "Energy",
    "XLF": "Financials",
    "XLV": "Health Care",
    "XLI": "Industrials",
    "XLB": "Materials",
    "XLRE": "Real Estate",
    "XLK": "Technology",
    "XLU": "Utilities",
}

tickers = list(sectors.keys()) + ["SPY", "QQQ"]
prices = yf.download(tickers, start="2018-01-01", auto_adjust=True, progress=False)["Close"].dropna(how="all")
prices.tail()

## 2. Calculate breadth metrics

I define breadth as the percentage of sector ETFs trading above their own moving averages.

This is simple, but useful:

- broad participation = many sectors above trend
- narrow participation = only a few sectors above trend

In [ ]:
sector_prices = prices[list(sectors.keys())]
above_50 = sector_prices > sector_prices.rolling(50).mean()
above_200 = sector_prices > sector_prices.rolling(200).mean()

breadth = pd.DataFrame(index=sector_prices.index)
breadth["pct_sectors_above_50d"] = above_50.mean(axis=1)
breadth["pct_sectors_above_200d"] = above_200.mean(axis=1)
breadth["spy"] = prices["SPY"]
breadth["qqq"] = prices["QQQ"]
breadth.tail()

In [ ]:
px.line(breadth[["pct_sectors_above_50d", "pct_sectors_above_200d"]], title="Sector breadth: % sectors above moving average").show()
px.line(breadth[["spy", "qqq"]], title="SPY and QQQ price context").show()

## 3. Identify sector leadership

A trader may want to know which sectors are leading over the last 1, 3 and 6 months.

In [ ]:
leaderboard = pd.DataFrame(index=list(sectors.keys()))
leaderboard["sector"] = pd.Series(sectors)
leaderboard["return_1m"] = sector_prices.pct_change(21).iloc[-1]
leaderboard["return_3m"] = sector_prices.pct_change(63).iloc[-1]
leaderboard["return_6m"] = sector_prices.pct_change(126).iloc[-1]
leaderboard["above_50d"] = above_50.iloc[-1]
leaderboard["above_200d"] = above_200.iloc[-1]
leaderboard = leaderboard.sort_values("return_3m", ascending=False)
leaderboard

In [ ]:
plot_df = leaderboard.reset_index().rename(columns={"index": "ticker"})
px.bar(plot_df, x="ticker", y="return_3m", color="above_50d", hover_data=["sector", "return_1m", "return_6m"], title="Sector leadership: 3-month returns").show()

## 4. My conclusion

This notebook demonstrates how to convert a market-internals question into a practical dashboard.

## Next iterations

- Add individual S&P 500 constituents if data access is available
- Add new highs / new lows
- Add advance-decline measures
- Add relative strength versus SPY
- Convert to Streamlit